# Task 5 — Deployment: Bias Detection and Mitigation

This notebook investigates a hotel-side fairness issue: whether the trained ranker underexposes independent hotels compared with branded hotels.

The approach is:
1. Train the original LightGBM LambdaRank model on the training fold.
2. Detect bias on the validation fold using exposure and ranking metrics.
3. Apply pre-processing re-weighting to give more importance to relevant independent hotels.
4. Retrain the model and compare original versus mitigated results.

In [ ]:
# ---------------------------------------------------
# Cell 1: Imports and general settings
# ---------------------------------------------------

import numpy as np
import pandas as pd
import lightgbm as lgb

import matplotlib.pyplot as plt

RANDOM_STATE = 42

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

In [ ]:
# ---------------------------------------------------
# Cell 2: Load prepared feature data from Task 3
# ---------------------------------------------------
# This notebook assumes that Task 3 already created:
# - data/train_features.parquet
# - data/val_features.parquet

train_feat = pd.read_parquet("data/train_features.parquet")
val_feat = pd.read_parquet("data/val_features.parquet")

print("Train feature shape:", train_feat.shape)
print("Validation feature shape:", val_feat.shape)

display(train_feat.head())

In [ ]:
# ---------------------------------------------------
# Cell 3: Define model feature columns
# ---------------------------------------------------
# These columns are not used as model inputs.
# srch_id and prop_id are identifiers.
# click_bool, booking_bool, gross_bookings_usd, position, and relevance are target-related or unavailable in test.
# date_time is not used directly because date/time features were already extracted in Task 3.

NON_FEATURE_COLS = [
    "srch_id",
    "prop_id",
    "date_time",
    "click_bool",
    "booking_bool",
    "gross_bookings_usd",
    "position",
    "relevance"
]

feature_cols = [
    col for col in train_feat.columns
    if col not in NON_FEATURE_COLS
    and pd.api.types.is_numeric_dtype(train_feat[col])
]

missing_in_val = [col for col in feature_cols if col not in val_feat.columns]

print("Number of feature columns:", len(feature_cols))
print("Missing features in validation:", missing_in_val)

if len(missing_in_val) > 0:
    raise ValueError("Some training features are missing in the validation set.")

In [ ]:
# ---------------------------------------------------
# Cell 4: Sort data and create LightGBM matrices/groups
# ---------------------------------------------------
# LightGBM ranker needs rows sorted by srch_id so that group sizes match consecutive rows.

train_feat = train_feat.sort_values("srch_id").reset_index(drop=True)
val_feat = val_feat.sort_values("srch_id").reset_index(drop=True)

X_train = train_feat[feature_cols]
y_train = train_feat["relevance"].astype(int)

X_val = val_feat[feature_cols]
y_val = val_feat["relevance"].astype(int)

group_train = train_feat.groupby("srch_id").size().to_numpy()
group_val = val_feat.groupby("srch_id").size().to_numpy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("Number of train groups:", len(group_train))
print("Number of validation groups:", len(group_val))
print("First 10 train group sizes:", group_train[:10])

In [ ]:
# ---------------------------------------------------
# Cell 5: Define ranking evaluation functions
# ---------------------------------------------------

def dcg_at_k(relevances, k=5):
    """Compute DCG@k for one ranked list."""
    relevances = np.asarray(relevances)[:k]
    if len(relevances) == 0:
        return 0.0

    discounts = np.log2(np.arange(2, len(relevances) + 2))
    gains = (2 ** relevances - 1)
    return np.sum(gains / discounts)


def ndcg_at_k_for_group(y_true, y_score, k=5):
    """Compute NDCG@k for one search group."""
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    order = np.argsort(-y_score)
    ranked_relevance = y_true[order]

    ideal_order = np.argsort(-y_true)
    ideal_relevance = y_true[ideal_order]

    dcg = dcg_at_k(ranked_relevance, k=k)
    idcg = dcg_at_k(ideal_relevance, k=k)

    if idcg == 0:
        return 0.0

    return dcg / idcg


def mean_ndcg_at_k(df, y_true_col="relevance", y_score_col="score", group_col="srch_id", k=5):
    """Compute mean NDCG@k over all search groups."""
    scores = []

    for _, group in df.groupby(group_col, sort=False):
        y_true = group[y_true_col].to_numpy()
        y_score = group[y_score_col].to_numpy()
        scores.append(ndcg_at_k_for_group(y_true, y_score, k=k))

    return float(np.mean(scores))

In [ ]:
# ---------------------------------------------------
# Cell 6: Define fairness / exposure evaluation functions
# ---------------------------------------------------
# We focus on hotel-side fairness:
# Are independent hotels underexposed in the model's top-5 rankings?

def add_rank_within_search(df, score_col="score"):
    """Add rank within each search, where rank 1 is the highest predicted score."""
    df = df.copy()
    df = df.sort_values(["srch_id", score_col], ascending=[True, False])
    df["pred_rank"] = df.groupby("srch_id").cumcount() + 1
    return df


def exposure_metrics_by_brand(df, score_col="score", k=5):
    """
    Compute top-k exposure metrics for independent and branded hotels.

    prop_brand_bool:
    - 0 = independent hotel
    - 1 = branded / major chain hotel
    """
    ranked = add_rank_within_search(df, score_col=score_col)
    topk = ranked[ranked["pred_rank"] <= k].copy()

    candidate_independent_share = (ranked["prop_brand_bool"] == 0).mean()
    candidate_branded_share = (ranked["prop_brand_bool"] == 1).mean()

    topk_independent_share = (topk["prop_brand_bool"] == 0).mean()
    topk_branded_share = (topk["prop_brand_bool"] == 1).mean()

    independent_exposure_gap = topk_independent_share - candidate_independent_share
    branded_exposure_gap = topk_branded_share - candidate_branded_share

    # Ratio < 1 means the group receives less top-k exposure than its candidate-set presence.
    independent_representation_ratio = topk_independent_share / candidate_independent_share if candidate_independent_share > 0 else np.nan
    branded_representation_ratio = topk_branded_share / candidate_branded_share if candidate_branded_share > 0 else np.nan

    metrics = {
        "candidate_independent_share": candidate_independent_share,
        "topk_independent_share": topk_independent_share,
        "independent_exposure_gap": independent_exposure_gap,
        "independent_representation_ratio": independent_representation_ratio,

        "candidate_branded_share": candidate_branded_share,
        "topk_branded_share": topk_branded_share,
        "branded_exposure_gap": branded_exposure_gap,
        "branded_representation_ratio": branded_representation_ratio,
    }

    return metrics


def booked_recall_at_k_by_brand(df, score_col="score", k=5):
    """
    Compute how often the booked hotel is placed in the top-k,
    separately for independent and branded booked hotels.
    """
    ranked = add_rank_within_search(df, score_col=score_col)

    booked = ranked[ranked["booking_bool"] == 1].copy()

    results = {}

    for brand_value, group_name in [(0, "independent"), (1, "branded")]:
        group_booked = booked[booked["prop_brand_bool"] == brand_value]

        if len(group_booked) == 0:
            results[f"booked_recall@{k}_{group_name}"] = np.nan
            results[f"num_booked_{group_name}"] = 0
        else:
            results[f"booked_recall@{k}_{group_name}"] = (group_booked["pred_rank"] <= k).mean()
            results[f"num_booked_{group_name}"] = len(group_booked)

    return results


def mean_ndcg_at_k_by_booked_brand(df, score_col="score", k=5):
    """
    Compute mean NDCG@k separately for searches where the booked hotel is independent
    versus searches where the booked hotel is branded.

    Searches without a booking are excluded from this subgroup metric.
    """
    booked_rows = df[df["booking_bool"] == 1][["srch_id", "prop_brand_bool"]].copy()
    booked_rows = booked_rows.rename(columns={"prop_brand_bool": "booked_hotel_brand_bool"})

    # A search should normally have at most one booked property.
    booked_rows = booked_rows.drop_duplicates("srch_id")

    df_with_group = df.merge(booked_rows, on="srch_id", how="inner")

    results = {}

    for brand_value, group_name in [(0, "booked_independent"), (1, "booked_branded")]:
        subset = df_with_group[df_with_group["booked_hotel_brand_bool"] == brand_value]

        if subset["srch_id"].nunique() == 0:
            results[f"ndcg@{k}_{group_name}_searches"] = np.nan
            results[f"num_{group_name}_searches"] = 0
        else:
            results[f"ndcg@{k}_{group_name}_searches"] = mean_ndcg_at_k(
                subset,
                y_true_col="relevance",
                y_score_col=score_col,
                group_col="srch_id",
                k=k
            )
            results[f"num_{group_name}_searches"] = subset["srch_id"].nunique()

    return results


def full_task5_evaluation(df, model_name, score_col="score", k=5):
    """Combine ranking quality and fairness metrics into one dictionary."""
    metrics = {
        "model": model_name,
        f"overall_ndcg@{k}": mean_ndcg_at_k(
            df,
            y_true_col="relevance",
            y_score_col=score_col,
            group_col="srch_id",
            k=k
        )
    }

    metrics.update(exposure_metrics_by_brand(df, score_col=score_col, k=k))
    metrics.update(booked_recall_at_k_by_brand(df, score_col=score_col, k=k))
    metrics.update(mean_ndcg_at_k_by_booked_brand(df, score_col=score_col, k=k))

    return metrics

In [ ]:
# ---------------------------------------------------
# Cell 7: Train original best model from Task 4
# ---------------------------------------------------
# These are the best parameters from your Task 4 tuning.

best_parameters = {
    "num_leaves": 383,
    "learning_rate": 0.02,
    "min_child_samples": 900,
    "reg_lambda": 10.0,
    "reg_alpha": 0.0,
    "subsample": 0.9,
    "colsample_bytree": 0.8,
    "n_estimators": 286
}

original_ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    ndcg_eval_at=[5],
    boosting_type="gbdt",

    n_estimators=best_parameters["n_estimators"],
    learning_rate=best_parameters["learning_rate"],
    num_leaves=best_parameters["num_leaves"],
    max_depth=-1,
    min_child_samples=best_parameters["min_child_samples"],

    reg_lambda=best_parameters["reg_lambda"],
    reg_alpha=best_parameters["reg_alpha"],
    subsample=best_parameters["subsample"],
    colsample_bytree=best_parameters["colsample_bytree"],

    random_state=RANDOM_STATE,
    n_jobs=-1
)

original_ranker.fit(
    X_train,
    y_train,
    group=group_train,

    eval_set=[(X_val, y_val)],
    eval_group=[group_val],
    eval_at=[5],

    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100)
    ]
)

print("Original model best iteration:", original_ranker.best_iteration_)

In [ ]:
# ---------------------------------------------------
# Cell 8: Evaluate original model for ranking quality and bias
# ---------------------------------------------------

original_val_scores = original_ranker.predict(
    X_val,
    num_iteration=original_ranker.best_iteration_
)

original_eval_df = val_feat[[
    "srch_id",
    "prop_id",
    "prop_brand_bool",
    "relevance",
    "booking_bool",
    "click_bool"
]].copy()

original_eval_df["score"] = original_val_scores

original_metrics = full_task5_evaluation(
    original_eval_df,
    model_name="Original LightGBM Ranker",
    score_col="score",
    k=5
)

original_metrics_df = pd.DataFrame([original_metrics])

display(original_metrics_df.T.rename(columns={0: "value"}))

In [ ]:
# ---------------------------------------------------
# Cell 9: Inspect original top-5 exposure more intuitively
# ---------------------------------------------------

ranked_original = add_rank_within_search(original_eval_df, score_col="score")
top5_original = ranked_original[ranked_original["pred_rank"] <= 5].copy()

brand_exposure_original = pd.DataFrame({
    "group": ["Independent hotels", "Branded hotels"],
    "candidate_share": [
        (ranked_original["prop_brand_bool"] == 0).mean(),
        (ranked_original["prop_brand_bool"] == 1).mean()
    ],
    "top5_share": [
        (top5_original["prop_brand_bool"] == 0).mean(),
        (top5_original["prop_brand_bool"] == 1).mean()
    ]
})

brand_exposure_original["exposure_gap"] = (
    brand_exposure_original["top5_share"] -
    brand_exposure_original["candidate_share"]
)

brand_exposure_original["representation_ratio"] = (
    brand_exposure_original["top5_share"] /
    brand_exposure_original["candidate_share"]
)

display(brand_exposure_original)

In [ ]:
# ---------------------------------------------------
# Cell 10: Plot original candidate share versus top-5 exposure share
# ---------------------------------------------------

plot_df = brand_exposure_original.set_index("group")[["candidate_share", "top5_share"]]

ax = plot_df.plot(kind="bar", figsize=(8, 5))
ax.set_title("Original model: candidate share vs. top-5 exposure share")
ax.set_ylabel("Share")
ax.set_xlabel("")
ax.legend(["Candidate share", "Top-5 exposure share"])
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------
# Cell 11: Create pre-processing re-weighting for mitigation
# ---------------------------------------------------
# Mitigation idea:
# - Keep all rows.
# - Give more training weight to independent hotels that were relevant.
# - Give an even higher weight to independent hotels that were booked.
#
# This aims to make the model pay more attention to independent hotels
# that users actually clicked or booked.

sample_weight_fair = np.ones(len(train_feat), dtype=np.float32)

# Relevant independent hotels: click-only or booking.
sample_weight_fair[
    (train_feat["prop_brand_bool"] == 0) &
    (train_feat["relevance"] > 0)
] = 1.5

# Booked independent hotels: strongest relevance signal.
sample_weight_fair[
    (train_feat["prop_brand_bool"] == 0) &
    (train_feat["relevance"] == 5)
] = 2.0

weight_summary = pd.DataFrame({
    "weight": sample_weight_fair
}).value_counts().reset_index(name="count")

weight_summary["share"] = weight_summary["count"] / len(sample_weight_fair)

display(weight_summary.sort_values("weight"))

In [ ]:
# ---------------------------------------------------
# Cell 12: Train mitigated model with re-weighting
# ---------------------------------------------------

fair_ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    ndcg_eval_at=[5],
    boosting_type="gbdt",

    n_estimators=best_parameters["n_estimators"],
    learning_rate=best_parameters["learning_rate"],
    num_leaves=best_parameters["num_leaves"],
    max_depth=-1,
    min_child_samples=best_parameters["min_child_samples"],

    reg_lambda=best_parameters["reg_lambda"],
    reg_alpha=best_parameters["reg_alpha"],
    subsample=best_parameters["subsample"],
    colsample_bytree=best_parameters["colsample_bytree"],

    random_state=RANDOM_STATE,
    n_jobs=-1
)

fair_ranker.fit(
    X_train,
    y_train,
    group=group_train,
    sample_weight=sample_weight_fair,

    eval_set=[(X_val, y_val)],
    eval_group=[group_val],
    eval_at=[5],

    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100)
    ]
)

print("Mitigated model best iteration:", fair_ranker.best_iteration_)

In [ ]:
# ---------------------------------------------------
# Cell 13: Evaluate mitigated model
# ---------------------------------------------------

fair_val_scores = fair_ranker.predict(
    X_val,
    num_iteration=fair_ranker.best_iteration_
)

fair_eval_df = val_feat[[
    "srch_id",
    "prop_id",
    "prop_brand_bool",
    "relevance",
    "booking_bool",
    "click_bool"
]].copy()

fair_eval_df["score"] = fair_val_scores

fair_metrics = full_task5_evaluation(
    fair_eval_df,
    model_name="Re-weighted LightGBM Ranker",
    score_col="score",
    k=5
)

fair_metrics_df = pd.DataFrame([fair_metrics])

display(fair_metrics_df.T.rename(columns={0: "value"}))

In [ ]:
# ---------------------------------------------------
# Cell 14: Compare original and mitigated model
# ---------------------------------------------------

comparison_df = pd.DataFrame([original_metrics, fair_metrics])

# Put model name first and round numeric values for readability.
comparison_display = comparison_df.copy()

for col in comparison_display.columns:
    if col != "model":
        comparison_display[col] = pd.to_numeric(comparison_display[col], errors="ignore")

display(comparison_display)
display(comparison_display.T)

In [ ]:
# ---------------------------------------------------
# Cell 15: Make a compact report table
# ---------------------------------------------------
# This table contains the most important results for the report.

important_cols = [
    "model",
    "overall_ndcg@5",

    "candidate_independent_share",
    "topk_independent_share",
    "independent_exposure_gap",
    "independent_representation_ratio",

    "booked_recall@5_independent",
    "booked_recall@5_branded",

    "ndcg@5_booked_independent_searches",
    "ndcg@5_booked_branded_searches"
]

report_table = comparison_df[important_cols].copy()

numeric_cols = [col for col in report_table.columns if col != "model"]
report_table[numeric_cols] = report_table[numeric_cols].astype(float).round(6)

display(report_table)

report_table.to_csv("task5_bias_mitigation_results.csv", index=False)

print("Saved report table to: task5_bias_mitigation_results.csv")

In [ ]:
# ---------------------------------------------------
# Cell 16: Plot comparison of independent hotel exposure
# ---------------------------------------------------

exposure_comparison = comparison_df[[
    "model",
    "candidate_independent_share",
    "topk_independent_share"
]].copy()

exposure_comparison = exposure_comparison.set_index("model")

ax = exposure_comparison.plot(kind="bar", figsize=(9, 5))
ax.set_title("Independent hotels: candidate share vs. top-5 exposure share")
ax.set_ylabel("Share")
ax.set_xlabel("")
ax.legend(["Candidate share", "Top-5 exposure share"])
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------
# Cell 17: Plot comparison of booked hotel recall@5 by brand group
# ---------------------------------------------------

recall_comparison = comparison_df[[
    "model",
    "booked_recall@5_independent",
    "booked_recall@5_branded"
]].copy()

recall_comparison = recall_comparison.set_index("model")

ax = recall_comparison.plot(kind="bar", figsize=(9, 5))
ax.set_title("Booked-hotel Recall@5 by hotel brand group")
ax.set_ylabel("Recall@5")
ax.set_xlabel("")
ax.legend(["Independent booked hotels", "Branded booked hotels"])
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------
# Cell 18: Optional sensitivity experiment with stronger weights
# ---------------------------------------------------
# Only run this cell if the first mitigation is too weak.
# It trains a second mitigated model with stronger weights:
# - relevant independent hotels: 2.0
# - booked independent hotels: 3.0
#
# If you are short on time, you can skip this cell.

RUN_STRONGER_WEIGHT_EXPERIMENT = False

if RUN_STRONGER_WEIGHT_EXPERIMENT:
    sample_weight_stronger = np.ones(len(train_feat), dtype=np.float32)

    sample_weight_stronger[
        (train_feat["prop_brand_bool"] == 0) &
        (train_feat["relevance"] > 0)
    ] = 2.0

    sample_weight_stronger[
        (train_feat["prop_brand_bool"] == 0) &
        (train_feat["relevance"] == 5)
    ] = 3.0

    stronger_ranker = lgb.LGBMRanker(
        objective="lambdarank",
        metric="ndcg",
        ndcg_eval_at=[5],
        boosting_type="gbdt",

        n_estimators=best_parameters["n_estimators"],
        learning_rate=best_parameters["learning_rate"],
        num_leaves=best_parameters["num_leaves"],
        max_depth=-1,
        min_child_samples=best_parameters["min_child_samples"],

        reg_lambda=best_parameters["reg_lambda"],
        reg_alpha=best_parameters["reg_alpha"],
        subsample=best_parameters["subsample"],
        colsample_bytree=best_parameters["colsample_bytree"],

        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    stronger_ranker.fit(
        X_train,
        y_train,
        group=group_train,
        sample_weight=sample_weight_stronger,

        eval_set=[(X_val, y_val)],
        eval_group=[group_val],
        eval_at=[5],

        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=100)
        ]
    )

    stronger_val_scores = stronger_ranker.predict(
        X_val,
        num_iteration=stronger_ranker.best_iteration_
    )

    stronger_eval_df = val_feat[[
        "srch_id",
        "prop_id",
        "prop_brand_bool",
        "relevance",
        "booking_bool",
        "click_bool"
    ]].copy()

    stronger_eval_df["score"] = stronger_val_scores

    stronger_metrics = full_task5_evaluation(
        stronger_eval_df,
        model_name="Stronger re-weighted LightGBM Ranker",
        score_col="score",
        k=5
    )

    sensitivity_df = pd.DataFrame([original_metrics, fair_metrics, stronger_metrics])
    display(sensitivity_df[important_cols].round(6))

In [ ]:
# ---------------------------------------------------
# Cell 19: Save validation predictions for possible further analysis
# ---------------------------------------------------

original_eval_df.to_parquet("task5_original_validation_predictions.parquet", index=False)
fair_eval_df.to_parquet("task5_reweighted_validation_predictions.parquet", index=False)

print("Saved:")
print("- task5_original_validation_predictions.parquet")
print("- task5_reweighted_validation_predictions.parquet")

## How to interpret the results

Use the compact table from Cell 15 in the report.

A good outcome is not necessarily that every fairness metric becomes perfectly equal. Instead, describe the trade-off:

- Did the independent hotel top-5 exposure share move closer to its candidate share?
- Did booked independent hotel Recall@5 improve?
- How much did overall NDCG@5 change?
- Did performance for branded hotels remain acceptable?

A careful conclusion could be:

> The original model showed underexposure of independent hotels if their share in the top-5 recommendations was lower than their share among all candidate hotels. We applied pre-processing re-weighting by increasing the training weight of relevant independent hotels, especially booked independent hotels. This improved independent-hotel exposure and/or booked-hotel Recall@5, while causing only a small change in overall NDCG@5. The mitigation therefore reduced the measured hotel-side bias, but it also illustrates a trade-off between ranking performance and fairness.